In [1]:
from app.data.etl import GetData
from app.data.feature_engineering import FeatureEngineering

In [2]:
get_data = GetData(file_path='app/data/raw_data/BD_ordenes.xlsx')
df = get_data.read_data()

In [3]:
df.head()

,respuesta,consumo_criticado,servicio,categoria,nivel_tension,estrato,localidad,funcion_analisis,calificacion,obs_lectura,periodicidad
0,1,0.0,701-ENERGÍA MDO REGULADO,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOPR - Calcular Consumo Penalizado de Ener...,5035-BAJO ENERGIA (<-50%),30-VARIACION NIVEL DE UTILIZACIÓN,1
1,1,420.0,701-ENERGÍA MDO REGULADO,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOPR - Calcular Consumo Penalizado de Ener...,5080-MUY ALTO (>500%),30-VARIACION NIVEL DE UTILIZACIÓN,1
2,1,99999.0,101-AGUA POTABLE,1-RESIDENCIAL,NaN,1.0,5088-BELLO,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),34-LECTURA MENOR,1
3,1,881.0,701-ENERGÍA MDO REGULADO,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),35-NO HAY JUSTIFICACION,1
4,1,99999.0,701-ENERGÍA MDO REGULADO,1-RESIDENCIAL,220.0,4.0,5001-MEDELLÍN,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),34-LECTURA MENOR,1


# Feature Engineering

In [4]:
feature_engineering = FeatureEngineering(df)
df_fe, target_variable, numeric_features, categorical_features = feature_engineering.prepare_features_only_energy()

In [5]:
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target variable:", target_variable)

Numeric features: ['consumo_criticado', 'nivel_tension', 'estrato', 'periodicidad']
Categorical features: ['categoria', 'localidad', 'funcion_analisis', 'calificacion', 'obs_lectura', 'tipo_servicio']
Target variable: respuesta


In [6]:
df_fe.head()

,respuesta,consumo_criticado,categoria,nivel_tension,estrato,localidad,funcion_analisis,calificacion,obs_lectura,periodicidad,tipo_servicio
0,0,0.0,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOPR - Calcular Consumo Penalizado de Ener...,5035-BAJO ENERGIA (<-50%),30-VARIACION NIVEL DE UTILIZACIÓN,1,Energia
1,0,420.0,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOPR - Calcular Consumo Penalizado de Ener...,5080-MUY ALTO (>500%),30-VARIACION NIVEL DE UTILIZACIÓN,1,Energia
3,0,881.0,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),35-NO HAY JUSTIFICACION,1,Energia
4,0,99999.0,1-RESIDENCIAL,220.0,4.0,5001-MEDELLÍN,CALCCOLE - Calcular Consumo por Lecturas,5080-MUY ALTO (>500%),34-LECTURA MENOR,1,Energia
7,0,260.0,3-INDUSTRIAL,220.0,NaN,5088-BELLO,CALCCOLE - Calcular Consumo por Lecturas,5035-BAJO ENERGIA (<-50%),30-VARIACION NIVEL DE UTILIZACIÓN,1,Energia


In [7]:
test_size = 0.2

# Modelo con MLFlow + Optuna

Antes de ejecutar, subir cliente de MLFlow con el comando.

* mlflow server --backend-store-uri sqlite:///consumos.db

In [8]:
from sklearn.ensemble import RandomForestClassifier
from app.train.train import TrainModel
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Define RandomForest parameter distributions
param_distributions = {
    'n_estimators': ('int', 50, 200),
    'max_depth': ('int', 5, 30),
    'min_samples_split': ('int', 2, 10),
    'min_samples_leaf': ('int', 1, 5),
    'max_features': ('categorical', ['sqrt', 'log2', None])
}

trainer = TrainModel(
    df_fe,
    numeric_features = numeric_features,
    categorical_features = categorical_features,
    target_column = target_variable,
    model_class = RandomForestClassifier,
    test_size = test_size,
    model_params = {'random_state': 42, 'n_jobs': -1},
    param_distributions = param_distributions,
    n_trials = 50,
    optimization_metric = 'f1',
    mlflow_setup = mlflow,
    mlflow_experiment_name = 'experimento_desviacion_consumos',
    mlflow_tracking_uri = 'http://localhost:5000'
)

best_pipeline, run_id, study = trainer.train()

[I 2025-10-18 17:20:34,513] A new study created in memory with name: optuna_RandomForestClassifier
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/src/app/train/train.py:187: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
INFO:app.train.train:Starting Optuna optimization with 50 trials...
INFO:app.train.train:Optimizing for: f1
INFO:app.train.train:Model type: RandomForestClassifier
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:20:35,512] Trial 0 finished with value: 0.8222175631706217 and parameters: {'n_estimators': 130, 'max_depth': 21, 'min_samples_split': 3, 'min_samples_leaf': 3, 'max_features': None}. Best

🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/1/runs/c5a494d24e5949739b46bdd28fafadf3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:20:36,827] Trial 1 finished with value: 0.8220859617125831 and parameters: {'n_estimators': 159, 'max_depth': 16, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 0 with value: 0.8222175631706217.


🏃 View run 1 at: http://127.0.0.1:5000/#/experiments/1/runs/39e92241c7b5425abd9c1ab273f27a24
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:20:37,133] Trial 2 finished with value: 0.8071757569079356 and parameters: {'n_estimators': 180, 'max_depth': 27, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.8222175631706217.
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:20:37,363] Trial 3 finished with value: 0.6024893001678888 and parameters: {'n_estimators': 164, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max

...

🏃 View run 48 at: http://127.0.0.1:5000/#/experiments/1/runs/ff9281c6246440ad862fd79810ad9aa5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2025-10-18 17:21:01,639] Trial 49 finished with value: 0.8070743070194649 and parameters: {'n_estimators': 125, 'max_depth': 26, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 43 with value: 0.8259298974644069.
INFO:app.train.train:Optimization complete!
INFO:app.train.train:Best f1: 0.8259
INFO:app.train.train:Best parameters: {'n_estimators': 62, 'max_depth': 16, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': None}


🏃 View run 49 at: http://127.0.0.1:5000/#/experiments/1/runs/61b100b086f847feba3f088a507fdac1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/jonatanlondonotaborda/Documents/Repositorios/Proyecto1_UdeM/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/do

🏃 View run best_model_RandomForestClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/e46bcbb331344e1483d3c416f09d54d1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [10]:
print(f"Best MLflow run ID: {run_id}")  
print(f"Best hyperparameters: {study.best_params}")

Best MLflow run ID: e46bcbb331344e1483d3c416f09d54d1
Best hyperparameters: {'n_estimators': 62, 'max_depth': 16, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': None}
